# Day 1 Practice (SOLUTION): Predict Titanic Survival

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rhythman/CS_Mines_summer_hs_intern/blob/main/day1/solutions/02_titanic_solution.ipynb)

This is the **fully worked answer key** for the Titanic practice notebook. Every blank is filled in, every figure is drawn and explained, and the whole thing runs top-to-bottom with **Kernel → Restart & Run All**. Use it to check your own work after you have honestly tried the practice version.

The *concepts* here (what a feature is, what overfitting means, why we hold out a test set, how a decision tree splits) all come from the lecture. This notebook is about turning those ideas into code and pictures. When a concept shows up, we point back to the lecture rather than re-teach it from scratch.

## The 10-step workflow

Almost every supervised-learning project follows the same shape. We will walk it in order:

1. **Load** the data.
2. **Explore** it with tables and plots (EDA).
3. **Clean** it: fill in or handle missing values.
4. **Encode** text columns into numbers.
5. **Engineer** new features from the ones we have.
6. **Select** which columns to feed the model.
7. **Split** into training and validation sets.
8. **Train** several models and compare them.
9. **Evaluate** the best one honestly.
10. **Predict** on new data and save a submission file.

## Three datasets, one golden rule

- **`train.csv`** (891 passengers) has the answer column `Survived`. This is the only data with labels, so it is *all* we can learn from and *all* we can honestly score on.
- The **validation slice** is a piece we carve out of `train.csv` and hide from the model, so we can measure performance on passengers the model has never seen.
- **`test.csv`** (418 passengers) has **no** `Survived` column. It exists only so you can submit predictions to the Kaggle leaderboard; we can never check our own accuracy on it.

> ⭐ **Golden rule.** Split the data *before* you compute anything from it (medians, encoders, and so on). Any number the model uses must come from the training split only. Peeking at validation or test data while preparing features is called *data leakage*, and it makes your accuracy look great in the notebook and terrible in the real world.

## Step 1: Setup and Load

The first code cell is the **setup cell**. It imports every library we need, fixes the random seed so results are repeatable, sets a clean plot style, and defines `load_csv(...)` which finds the data whether you are on your laptop or in Google Colab. Run it once, first, every session.

In [ ]:
# ============================================================================
# CANONICAL SETUP CELL  (paste verbatim as the first CODE cell in every notebook)
# ============================================================================
# Standard library
import os
import random

# Third-party scientific stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# scikit-learn pieces we use across the workflow
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Show plots INSIDE the notebook (this is a Jupyter "magic", not normal Python).
%matplotlib inline

# --- Reproducibility: one seed, used everywhere ------------------------------
# Setting the seed makes "random" steps (shuffling, model training) repeatable,
# so you and your neighbor get the SAME numbers. Change nothing about the data
# and you get identical results every run.
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# --- Consistent, readable plots ---------------------------------------------
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.figsize"] = (8, 5)   # a comfortable default size
plt.rcParams["font.size"] = 12             # readable labels

# --- Are we on Google Colab or a local machine? -----------------------------
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

# --- Portable data loader: works in Colab AND local Jupyter ------------------
# It first looks for the bundled CSV next to the repo (data/ folder), and if it
# can't find it (e.g. a bare notebook uploaded to Colab), it downloads the same
# file from the public GitHub repo. Either way you get the identical data.
_DATA_CANDIDATES = ["../data", "data", "day1/data", "../day1/data", "."]
_RAW_BASE = "https://raw.githubusercontent.com/Rhythman/CS_Mines_summer_hs_intern/main/day1/data"

def load_csv(filename):
    """Load a bundled CSV by name, trying local paths first then the repo URL."""
    for folder in _DATA_CANDIDATES:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            print(f"Loaded '{filename}' from local file: {path}")
            return pd.read_csv(path)
    url = f"{_RAW_BASE}/{filename}"
    print(f"No local copy found - downloading '{filename}' from:\n  {url}")
    return pd.read_csv(url)

# Confirm the environment so students see it worked.
print("Environment:", "Google Colab" if ON_COLAB else "local Jupyter")
print("Versions -> pandas", pd.__version__, "| numpy", np.__version__, "| seaborn", sns.__version__)
import sklearn
print("           scikit-learn", sklearn.__version__)
print("Setup complete. RANDOM_STATE =", RANDOM_STATE)

Now load the two files. `load_csv` returns a **DataFrame**, pandas' table type, like a spreadsheet with named columns. We keep the raw tables in `train` and `test` and never overwrite them, so we can always go back to the untouched data.

In [ ]:
# Read the two CSV files into DataFrames.
train = load_csv("train.csv")   # 891 passengers, INCLUDES the Survived answer
test = load_csv("test.csv")     # 418 passengers, NO Survived column

# .shape is (rows, columns). Printing it confirms the files loaded correctly.
print("train shape:", train.shape)   # expect (891, 12)
print("test  shape:", test.shape)    # expect (418, 11)  <- one fewer column: Survived

`.head()` shows the first five rows so we can eyeball the columns. Each **row is one passenger**; each **column is one clue** (feature) about them. `Survived` is the **label**: `1` = survived, `0` = died.

In [ ]:
# Show the first 5 rows. In Jupyter, the LAST expression in a cell is displayed
# as a nice table automatically (no print needed).
train.head()

`.info()` lists every column, its data type, and how many **non-null** (present) values it has. Where the non-null count is below 891, that column has **missing values**, something we must handle before modeling.

In [ ]:
# Column names, dtypes, and non-null counts. Watch Age, Cabin, and Embarked.
train.info()

`.describe(include="all")` gives summary statistics. For **number** columns you get mean, min, max, and quartiles; for **text** columns you get counts and the most frequent value. This is a fast way to spot the scale of each feature (e.g. `Fare` ranges from 0 to 512).

In [ ]:
# Summary stats for every column, numeric and text alike.
train.describe(include="all")

Let's quantify the missing data directly. `.isnull()` turns the table into `True`/`False` (missing or not), and `.sum()` counts the `True`s per column.

In [ ]:
# Count missing values in each column of the training data.
train.isnull().sum()

> 🔎 **What to observe**
> - `Age` is missing for **177** passengers (about 20%). That is too many to throw away, so we will fill them in.
> - `Cabin` is missing for **687** passengers (about 77%). It is mostly empty, so we will not use it as-is.
> - `Embarked` is missing for just **2** passengers, an easy fix.
> - Every other column, including `Fare`, is complete in `train`.
>
> So `Age` needs careful filling, `Cabin` is too sparse to trust directly (we will turn it into a simple "had a cabin?" flag), and `Embarked`'s two gaps are trivial. In `test.csv`, `Fare` is also missing for one passenger, which we handle in Step 10.

## Step 2: Explore the data (EDA)

**EDA** = Exploratory Data Analysis: looking at the data with tables and plots *before* modeling, to learn which clues matter. We explore the raw `train` table here. Exploring is not the same as fitting a model, so looking at all of `train` is fine. Just notice that we do **not** change `train`; every plot reads from it without modifying it.

Each figure below is followed by a short "what to observe" note that ties the picture to a modeling decision.

### Figure 1: How many survived?

`sns.countplot` draws a bar for each value of a column and makes the bar as tall as the number of rows with that value. Here it counts how many passengers died (`0`) vs survived (`1`).

In [ ]:
# Count of passengers by outcome. countplot = "how many rows have each value?"
sns.countplot(data=train, x="Survived")
plt.title("How many passengers survived? (0 = died, 1 = survived)")
plt.xlabel("Survived")
plt.ylabel("Number of passengers")
plt.tight_layout()
plt.show()

> 🔎 **Read the chart**
> - The `0` (died) bar is clearly taller: **549 died** versus **342 survived**.
> - Survivors are about 38% of passengers, so the two classes are uneven but not wildly so.
>
> Because most people died, a lazy model that always predicts "everyone dies" is right about 62% of the time. That 61.6% is the baseline any real model has to beat.

### Figure 2: Did sex matter? (counts vs rate)

We look at `Sex` two ways. First, **counts** with `countplot` and `hue="Survived"`: `hue` splits each bar by the outcome, so we see raw numbers of survivors and non-survivors within each sex.

In [ ]:
# Raw counts: within each sex, how many died vs survived.
sns.countplot(data=train, x="Sex", hue="Survived")
plt.title("Survival counts by sex (raw numbers)")
plt.xlabel("Sex")
plt.ylabel("Number of passengers")
plt.legend(title="Survived", labels=["Died (0)", "Survived (1)"])
plt.tight_layout()
plt.show()

Counts can mislead when the groups are different sizes, so the **survival rate** is easier to compare. Because `Survived` is coded `0`/`1`, the average of that column is exactly the survival rate (a group with values `1, 0, 1` averages `0.67`, i.e. 67% survived). `sns.barplot` with `y="Survived"` computes that mean per group automatically. The thin black line on each bar is a confidence interval, a hint at how sure we are.

In [ ]:
# Survival RATE by sex. barplot shows the mean of y for each x group.
sns.barplot(data=train, x="Sex", y="Survived")
plt.title("Survival rate by sex")
plt.xlabel("Sex")
plt.ylabel("Survival rate")   # plain English, NOT "mean Survived"
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

> 🔎 **What jumps out**
> - In raw counts, far more men died than survived; for women it is the reverse.
> - As a rate, about **74% of women survived** versus only about **19% of men**, a huge gap.
> - The count plot and the rate plot tell the same story, but the rate makes the size of the effect obvious.
>
> `Sex` is the single strongest predictor of survival. Just guessing "all women live, all men die" would already be right about 79% of the time, a much tougher baseline to beat than 62%.

> ✅ **Check your understanding.** Why did we prefer the *rate* barplot over the *count* plot to judge how much sex matters?
> <details><summary>Show answer</summary>
>
> There were many more men than women aboard, so raw counts are affected by group size. A **rate** (average of the 0/1 `Survived` column) divides out the group size, letting us compare "chance of surviving" fairly across groups.
> </details>

### Figure 3: Did ticket class matter?

`Pclass` is passenger class: 1st, 2nd, or 3rd. We plot the survival **rate** per class, and then split by sex with `sns.catplot` (a `barplot` that can make side-by-side panels or grouped bars via `hue`).

In [ ]:
# Survival rate by passenger class.
sns.barplot(data=train, x="Pclass", y="Survived")
plt.title("Survival rate by passenger class")
plt.xlabel("Passenger class (1 = first, 3 = third)")
plt.ylabel("Survival rate")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# Same idea, now also split by sex. catplot with hue draws grouped bars.
# catplot returns its own figure, so we don't call plt.subplots ourselves.
g = sns.catplot(data=train, x="Pclass", y="Survived", hue="Sex",
                kind="bar", height=5, aspect=1.3)
g.set_axis_labels("Passenger class", "Survival rate")
g.figure.suptitle("Survival rate by class and sex", y=1.03)
plt.show()

> 🔎 **Notice the trend**
> - Survival falls steadily with class: **1st ~63%**, **2nd ~47%**, **3rd ~24%**, a clean downward trend.
> - Splitting by sex, women in 1st and 2nd class survived at very high rates, while men in 3rd class fared worst.
> - Class and sex both matter, and they stack: a 1st-class woman was in the safest group of all.
>
> `Pclass` is a strong, order-preserving predictor, so we keep it as a feature. Higher class meant better cabins, closer to the lifeboats, and more resources.

### Figure 4: Port of embarkation, and a confounder

`Embarked` is where a passenger boarded: **C** = Cherbourg, **Q** = Queenstown, **S** = Southampton. First the raw survival rate per port, then the same broken out by class.

In [ ]:
# Survival rate by port of embarkation.
sns.barplot(data=train, x="Embarked", y="Survived")
plt.title("Survival rate by port of embarkation")
plt.xlabel("Embarked (C=Cherbourg, Q=Queenstown, S=Southampton)")
plt.ylabel("Survival rate")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# Break the same plot out by class using facet columns (one panel per class).
g = sns.catplot(data=train, x="Embarked", y="Survived", col="Pclass",
                kind="bar", height=4, aspect=0.9)
g.set_axis_labels("Embarked", "Survival rate")
g.figure.suptitle("Survival rate by port, split by class", y=1.05)
plt.show()

> 🔎 **What to observe**
> - Overall, Cherbourg (**C ~55%**) looks safer than Queenstown (**Q ~39%**) and Southampton (**S ~34%**).
> - But once you split by class, the port gaps mostly shrink. Cherbourg simply loaded a larger share of 1st-class passengers.
> - So "boarded at C" is partly a stand-in for "rich."
>
> This is a **confounder**: the port looks predictive only because it correlates with class. We will still include `Embarked`, but now we understand *why* it appears to matter.

### Figure 5: Age

`sns.histplot` bins a numeric column and draws how many passengers fall in each bin. With `hue="Survived"` we overlay the died vs survived distributions to compare their shapes.

In [ ]:
# Age distribution, overlaid by outcome. multiple="layer" draws both on the same axes.
sns.histplot(data=train, x="Age", hue="Survived", bins=30, multiple="layer")
plt.title("Age distribution: survivors vs non-survivors")
plt.xlabel("Age (years)")
plt.ylabel("Number of passengers")
plt.tight_layout()
plt.show()

> 🔎 **Read the histogram**
> - Most passengers are 20-40 years old; the curve is bumpy, not a neat bell.
> - Among the youngest children (roughly age < 10) survivors clearly outnumber non-survivors, the "women and children first" effect.
> - Elsewhere the two curves overlap a lot, so age alone does not separate the outcomes cleanly.
> - Remember that about 20% of ages are **missing** and are not shown here at all.
>
> Age carries a real but **non-linear** signal: children survive more, but it is not simply "older is safer." Tree models can use that, and we will fill the missing ages before modeling.

### Figure 6: Fare (and why we take its log)

Fares are heavily **right-skewed**: most tickets are cheap, a few are enormous. When a column is that lopsided, plotting `log1p(fare)` = `log(1 + fare)` spreads the crowded low end out so patterns become visible. We draw both side by side.

In [ ]:
# Two panels: raw Fare (left) and log-scaled Fare (right).
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: the raw fares. Note the long, thin tail to the right.
sns.histplot(data=train, x="Fare", bins=40, ax=axes[0])
axes[0].set_title("Fare (raw) - heavily right-skewed")
axes[0].set_xlabel("Fare (£)")
axes[0].set_ylabel("Number of passengers")

# Right: log1p(Fare) makes the shape easier to read. We plot into a temp column.
train_fare_log = np.log1p(train["Fare"])   # log(1 + Fare); safe even when Fare == 0
sns.histplot(x=train_fare_log, bins=40, ax=axes[1])
axes[1].set_title("log(1 + Fare) - skew tamed")
axes[1].set_xlabel("log(1 + Fare)")
axes[1].set_ylabel("Number of passengers")

plt.tight_layout()
plt.show()

> 🔎 **What to observe**
> - Raw fares pile up near £0-£50 with a thin tail stretching past £500, which is hard to read.
> - After the log transform the distribution is roughly bell-shaped and easy to compare.
> - Higher fares track with higher classes, and we already saw that higher classes survived more.
>
> `Fare` carries a class-like survival signal. Tree models do not care about skew, so we keep `Fare` as-is, but the log view confirms it is informative.

### Figure 7: Where is the data missing?

A quick bar of missing counts per column makes the cleaning to-do list obvious. We sort so the worst offenders stand out.

In [ ]:
# Missing-value count per column, largest first. .plot(kind="bar") is pandas' own plotter.
missing = train.isnull().sum().sort_values(ascending=False)
missing.plot(kind="bar")
plt.title("Missing values per column (training data)")
plt.xlabel("Column")
plt.ylabel("Number of missing values")
plt.tight_layout()
plt.show()

> 🔎 **What stands out**
> - `Cabin` towers over everything (**687** missing), then `Age` (**177**), then `Embarked` (**2**).
> - Every other column has zero missing values in `train`.
>
> The cleaning plan almost writes itself: turn `Cabin` into a simple has-cabin flag, fill `Age` with a typical value, and fill the two `Embarked` gaps with the most common port.

### Figure 8: Correlation heatmap

A **correlation** measures how two numeric columns move together, from `-1` (opposite) through `0` (unrelated) to `+1` (in lockstep). `Sex` is text, so we make a temporary numeric copy (female = 1, male = 0) just for this plot; we do **not** touch the real `train`. `sns.heatmap` colors each correlation, and `center=0` puts white at zero so red = positive and blue = negative.

In [ ]:
# Build a throwaway numeric frame for correlations (does NOT modify train).
corr_df = train.copy()
corr_df["Sex"] = corr_df["Sex"].map({"male": 0, "female": 1})   # female = 1

# Keep only the numeric columns that are meaningful to correlate.
cols = ["Survived", "Sex", "Pclass", "Age", "SibSp", "Parch", "Fare"]
corr = corr_df[cols].corr()   # a 7x7 table of pairwise correlations

sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation between numeric features (and Survived)")
plt.tight_layout()
plt.show()

> 🔎 **Notice**
> - `Sex` (female = 1) has the strongest link to `Survived`: about **+0.54**.
> - `Pclass` is negative (about **-0.34**): a *bigger* class number (3rd) means *lower* survival.
> - `Fare` is positive (about **+0.26**), and `Fare` is itself strongly tied to `Pclass`, so they carry overlapping information.
> - `Age` sits near **0** here, even though the histogram showed children surviving more. Correlation only sees *straight-line* relationships, so it misses the children bump.
>
> Sex and class are our best numeric signals, but watch out for the trap below.

> ⚠️ **Pitfall: correlation is not causation.** `Fare` correlates with survival, but paying more did not *cause* anyone to live; a high fare is a **proxy** for first class, which came with better cabin locations and lifeboat access. Correlation also only catches straight-line patterns, which is why `Age` looks useless here despite the real children effect. Read these numbers as clues, not verdicts.

## Step 3: Clean the data (leakage-aware)

Now we start changing the data, so the **Golden Rule** kicks in.

> ⭐ **Golden rule.** Split *before* you compute any fill value. The median age, the median fare, and the most common port must be measured on the training split only, then applied unchanged to the validation slice (and later the test set). If we measured them on the whole table, information about the validation passengers would leak into how we prepare the training passengers.

So the very first cleaning action is the split. We carve **20%** of `train` into a validation slice. `stratify=train["Survived"]` keeps the same died/survived ratio in both pieces, and `random_state=RANDOM_STATE` makes the split identical every run.

> **Why we hide some passengers.** If you grade a model on the very passengers it studied, of course it looks brilliant; it already saw their answers. That score tells you nothing about new people. So we hide a random 20% of the labeled passengers (the validation set), let the model learn only from the other 80%, and grade it on the part it never saw. That grade is the honest one.
> Learn more: [scikit-learn: Cross-validation - evaluating estimator performance](https://scikit-learn.org/stable/modules/cross_validation.html)

In [ ]:
# Split the LABELED data into a training part and a hidden validation part.
# We do this FIRST, before computing any fill values, to avoid leakage.
train_df, val_df = train_test_split(
    train,
    test_size=0.2,               # 20% held out for honest scoring
    stratify=train["Survived"],  # keep the ~38% survival ratio in both parts
    random_state=RANDOM_STATE,   # reproducible split
)

print("train_df:", train_df.shape, " val_df:", val_df.shape)

Now measure the fill values from `train_df` only and store them in plain variables. We reuse these exact numbers everywhere (on the training rows, the validation rows, and the test rows), so all three splits are treated identically.

In [ ]:
# Fill values, learned from the TRAINING split ONLY.
age_median = train_df["Age"].median()        # a typical age (about 28.5)
fare_median = train_df["Fare"].median()      # a typical fare (about 14.45)
embarked_mode = train_df["Embarked"].mode()[0]  # the most common port ("S")

print("age_median   =", age_median)
print("fare_median  =", fare_median)
print("embarked_mode=", embarked_mode)

We wrap the actual cleaning in a small function `basic_clean` so we can apply the **exact same steps** to every split. It:

- fills missing `Age` and `Fare` with the train medians, and missing `Embarked` with the train mode,
- creates `HasCabin` (1 if a cabin was recorded, else 0), a usable signal salvaged from the mostly-empty `Cabin` column,
- returns a new DataFrame (it calls `.copy()` and never uses `inplace=True`), so re-running the cell is always safe.

In [ ]:
def basic_clean(df):
    """Fill missing values using the TRAIN-derived constants, add HasCabin.

    Returns a NEW DataFrame; the input is left untouched (restart-safe).
    """
    df = df.copy()
    # Reassign the column (no inplace=True) so re-running is idempotent.
    df["Age"] = df["Age"].fillna(age_median)
    df["Fare"] = df["Fare"].fillna(fare_median)
    df["Embarked"] = df["Embarked"].fillna(embarked_mode)
    # "Was a cabin recorded?" turns 77%-missing Cabin into a clean 0/1 flag.
    df["HasCabin"] = df["Cabin"].notnull().astype(int)
    return df

# Apply the SAME cleaning to both splits.
train_df = basic_clean(train_df)
val_df = basic_clean(val_df)

Sanity check: the columns we filled should now have zero missing values in both splits. (`Cabin` itself is still mostly empty, which is fine; we replaced it with `HasCabin` and will drop the raw column later.)

In [ ]:
# Confirm no missing values remain in the columns we care about.
check_cols = ["Age", "Fare", "Embarked"]
print("Missing in train_df:\n", train_df[check_cols].isnull().sum(), sep="")
print("\nMissing in val_df:\n", val_df[check_cols].isnull().sum(), sep="")

## Step 4: Encode text into numbers

Models do arithmetic, so every feature has to be numeric. `Sex` is the easy case: only two values, so we **map** them directly to `0`/`1`. We write a tiny function again so the same mapping is reused on every split.

In [ ]:
def encode_sex(df):
    """Map Sex text to numbers: male -> 0, female -> 1. Returns a new frame."""
    df = df.copy()
    df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
    return df

train_df = encode_sex(train_df)
val_df = encode_sex(val_df)

# Peek: Sex is now 0/1.
train_df[["Sex", "Survived"]].head()

`Embarked` has three unordered values (C, Q, S). Turning it into a single `0/1/2` column would falsely tell the model that "S is greater than C." Instead we **one-hot encode**: one new 0/1 column per port. `Embarked` is not our only text column, though. Step 5 will add a `Title` column that also needs one-hot encoding, so to do it once and consistently we one-hot encode at the *end of Step 5*, after every categorical feature exists. For now, just remember that `Embarked` still needs encoding.

> **Why one-hot instead of 1, 2, 3.** Models do arithmetic on their inputs, so text has to become numbers. For a column with no natural order, like the boarding port (C, Q, S), labeling them 1, 2, 3 would fool the model into thinking S is "greater than" C, which means nothing here. One-hot encoding sidesteps that by making a separate 0/1 column for each value (Embarked_C, Embarked_Q, Embarked_S), so no fake ranking sneaks in.
> Learn more: [scikit-learn: Encoding categorical features](https://scikit-learn.org/stable/modules/preprocessing.html#encoding-categorical-features)

## Step 5: Engineer new features

Feature engineering means building new columns that expose signal the raw data hides.

**Family size.** `SibSp` (siblings/spouses aboard) and `Parch` (parents/children aboard) each tell only part of the story. Adding them plus one (the passenger) gives `FamilySize`. From that we derive `IsAlone` (1 if travelling solo).

In [ ]:
def add_family_features(df):
    """Add FamilySize (SibSp + Parch + self) and IsAlone (1 if solo)."""
    df = df.copy()
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)   # True/False -> 1/0
    return df

train_df = add_family_features(train_df)
val_df = add_family_features(val_df)

Let's see whether family size relates to survival. We plot the survival **rate** for each `FamilySize` on the training split (which still has the `Survived` label).

In [ ]:
# Survival rate by family size, measured on the training split.
sns.barplot(data=train_df, x="FamilySize", y="Survived")
plt.title("Survival rate by family size")
plt.xlabel("Family size (self + siblings/spouses + parents/children)")
plt.ylabel("Survival rate")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

> 🔎 **Notice the pattern**
> - **Solo** travellers (size 1) survived only about 30% of the time.
> - **Small families** (size 2-4) did best, roughly 55%-72%.
> - **Large families** (5+) drop off sharply, and the tallest bars there rest on very few passengers (wide error bars).
>
> The relationship is **U-shaped**, not a straight line: being alone hurt, a small family helped, and a big family hurt again. `FamilySize` and `IsAlone` are worth keeping.

**Title from name.** Each `Name` contains a title (Mr, Mrs, Miss, Master, Dr, …) that encodes sex, rough age, and social status all at once. We pull it out with a regular expression: `str.extract(r" ([A-Za-z]+)\.")` grabs the word that sits between a space and a period (e.g. `Mr.`). Then we tidy the messy long tail:

- French variants: `Mlle` and `Ms` → `Miss`, `Mme` → `Mrs`.
- Every rare title (Dr, Rev, Col, Countess, …) → a single bucket `"Rare"`.

In [ ]:
def add_title(df):
    """Extract a cleaned Title from Name (Mr/Mrs/Miss/Master/Rare)."""
    df = df.copy()
    # Grab the word before the period, e.g. "Braund, Mr. Owen" -> "Mr".
    df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.")[0]
    # Merge equivalent French titles into the common English ones.
    df["Title"] = df["Title"].replace(["Mlle", "Ms"], "Miss").replace("Mme", "Mrs")
    # Anything that is not one of the four common titles becomes "Rare".
    common = ["Mr", "Mrs", "Miss", "Master"]
    df["Title"] = df["Title"].where(df["Title"].isin(common), "Rare")
    return df

train_df = add_title(train_df)
val_df = add_title(val_df)

# How many of each title in the training split?
train_df["Title"].value_counts()

Now the survival rate per title.

In [ ]:
# Survival rate by title, measured on the training split.
sns.barplot(data=train_df, x="Title", y="Survived")
plt.title("Survival rate by title")
plt.xlabel("Title (extracted from Name)")
plt.ylabel("Survival rate")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

> 🔎 **What to observe**
> - **Mrs (~79%)** and **Miss (~70%)** survived at high rates, and **Master** (young boys, ~58%) did well too.
> - **Mr (~16%)**, adult men, had by far the worst odds.
> - **Rare** titles land in the middle (~35%), on relatively few passengers.
>
> `Title` neatly bundles sex, age, and status into one feature, and it clearly separates outcomes. That makes it a strong engineered feature.

Now that `Embarked` and `Title` both exist, we finish the encoding with one-hot columns. `pd.get_dummies(..., columns=[...], dtype=int)` replaces each text column with one `0/1` column per category (`dtype=int` keeps them clean integers instead of `True`/`False`).

Because we encode each split separately, a category present in `train_df` might be absent in `val_df` (or the other way round), producing slightly different columns. We do **not** fix that here. We line the columns up in Step 6 with `reindex`, which is the standard way to guarantee that train and validation share the exact same feature layout.

In [ ]:
def one_hot(df):
    """One-hot encode the remaining text columns (Embarked, Title)."""
    return pd.get_dummies(df, columns=["Embarked", "Title"], dtype=int)

train_df = one_hot(train_df)
val_df = one_hot(val_df)

# The new dummy columns, e.g. Embarked_C, Embarked_Q, Embarked_S, Title_Mr, ...
print("train_df columns now:\n", list(train_df.columns))

## Step 6: Select the features

Not every column belongs in the model. We **drop**:

- `PassengerId`: a meaningless row number. A model could memorize IDs and "cheat" without learning anything general.
- `Name`, `Ticket`: free text and near-unique codes (**high cardinality**); we already squeezed the useful part of `Name` into `Title`.
- `Cabin`: 77% missing, and we kept its signal as `HasCabin`.
- `Survived`: that is the answer (`y`), not a feature.

> ⚠️ **Pitfall.** Never leave an identifier like `PassengerId` in the features. A flexible model can memorize "ID 806 survived" and score perfectly on data it has seen, then fail completely on new passengers. Dropping IDs forces it to learn real patterns.

We build `X` (features) and `y` (label) for each split, then `reindex` the validation features to match the training columns exactly, filling any missing dummy column with `0`.

In [ ]:
# Columns that must NOT be fed to the model.
drop_cols = ["PassengerId", "Survived", "Name", "Ticket", "Cabin"]

# Features (X) and label (y) for the training split.
X_train = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns])
y_train = train_df["Survived"]

# Same for the validation split...
X_val = val_df.drop(columns=[c for c in drop_cols if c in val_df.columns])
y_val = val_df["Survived"]

# ...then force X_val to have EXACTLY the training columns, in the same order.
# Any column missing in val (e.g. a rare dummy) is created and filled with 0.
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)

print("X_train:", X_train.shape, " X_val:", X_val.shape)
print("Feature columns:\n", list(X_train.columns))

## Step 7: Split? Already done. Meet the baseline.

In the 10-step roadmap this is the "Split" step, but the **Golden Rule** forced us to split all the way back in Step 3, *before* we computed a single median. That ordering is the whole point: split first, then learn fill values from the training part only. So our `X_train / y_train / X_val / y_val` are ready, and here we simply establish the baseline to beat.

A **baseline** is the score of a trivial rule. `DummyClassifier(strategy="most_frequent")` always predicts the most common class in the training data, here "died." If a real model cannot beat this, it has learned nothing.

> **Why we start with a deliberately dumb rule.** Before you can call an accuracy number "good," you need something to compare it against. The baseline is the laziest possible model: always guess the most common outcome ("died") and ignore every clue. Since about 62% of passengers died, that alone scores about 0.62. Any real model has to beat that bar, or it has learned nothing worth keeping.

In [ ]:
# The "everyone died" baseline. It ignores the features entirely.
baseline = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)                       # just learns the majority class
baseline_acc = accuracy_score(y_val, baseline.predict(X_val))
print(f"Baseline (always predict 'died') validation accuracy: {baseline_acc:.3f}")

> ⚠️ **Pitfall.** Always compare against a baseline before celebrating. About **62%** of passengers died, so "always guess died" already scores about 0.62. An accuracy of, say, 0.68 is barely better than doing nothing. Every model below has to clear this bar, and ideally beat the ~0.79 "all women survive" rule too.

## Step 8: Train several models and compare

We now fit three kinds of model on `X_train` and score each on the held-out `X_val`. We watch **training** accuracy and **validation** accuracy together: a model that scores far higher on training than validation is overfitting (memorizing, not generalizing).

> **What overfitting really is.** Overfitting means the model memorized the training passengers instead of learning general rules. You catch it by comparing two numbers: accuracy on the training data and accuracy on the held-out validation data. A model that scores 0.99 on training but 0.78 on validation obviously memorized. A healthy model scores about the same on both. That gap is exactly why we cap a tree's depth later on: a shallow tree cannot memorize, so it is forced to find broad rules that also hold for strangers.
> Learn more: [Google ML Crash Course: Overfitting](https://developers.google.com/machine-learning/crash-course/overfitting/overfitting), [StatQuest: Machine Learning Fundamentals - Bias and Variance](https://www.youtube.com/watch?v=EuBBz3bI-aA)

**Model 1: Logistic Regression.** A linear model that outputs a survival probability. `max_iter=1000` just gives it enough iterations to settle on an answer for this unscaled data.

> **In plain English: what logistic regression is.** The name has "regression" in it, but it is really a classifier. It hands each clue a weight (sex might earn a big positive weight, a higher class number a negative one), adds up the weighted clues for a passenger, and runs that total through an S-shaped curve that turns any number into a probability between 0 and 1. Above 0.5 it predicts "survived," below it predicts "died." It trains fast, and since you can read the weights, it is easy to interpret.
> Learn more: [StatQuest: Logistic Regression, Clearly Explained!!!](https://www.youtube.com/watch?v=yIYKR4sgzI8)

In [ ]:
# Model 1: Logistic Regression.
logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)  # max_iter=1000 gives the solver enough passes to settle (converge)
logreg.fit(X_train, y_train)
logreg_train = accuracy_score(y_train, logreg.predict(X_train))
logreg_val = accuracy_score(y_val, logreg.predict(X_val))
print(f"Logistic Regression  train={logreg_train:.3f}  val={logreg_val:.3f}")

**Model 2: Decision Tree.** A tree asks yes/no questions to split passengers into groups. First we grow an unrestricted tree to *see* overfitting: it memorizes the training data almost perfectly but scores much lower on validation.

> **In plain English: what a decision tree is.** A decision tree is a flowchart of yes/no questions. It scans the passengers and finds the single question that best separates survivors from non-survivors (say, "is the passenger female?"), then asks a follow-up question inside each group, and keeps splitting. Every split tries to make its two groups as "pure" as possible, mostly survivors on one side and mostly non-survivors on the other. Follow the answers down to the bottom box and that box is the tree's prediction. Because you can literally read the boxes, a shallow tree is one of the few models you can fully understand by eye.
> Learn more: [StatQuest: Decision and Classification Trees, Clearly Explained!!!](https://www.youtube.com/watch?v=_L39rN6gz7Y), [scikit-learn: Decision Trees](https://scikit-learn.org/stable/modules/tree.html)

In [ ]:
# Model 2a: an UNRESTRICTED tree that deliberately overfits, to show the gap.
tree_full = DecisionTreeClassifier(random_state=RANDOM_STATE)
tree_full.fit(X_train, y_train)
full_train = accuracy_score(y_train, tree_full.predict(X_train))
full_val = accuracy_score(y_val, tree_full.predict(X_val))
print(f"Unrestricted tree    train={full_train:.3f}  val={full_val:.3f}  <- big gap = overfitting")

Now a shallow tree (`max_depth=3`) can only ask three layers of questions, so it must find broad, general rules instead of memorizing. The train/validation gap shrinks. Because the tree is small, we can literally draw it with `plot_tree`.

In [ ]:
# Model 2b: a shallow tree that generalizes better.
tree = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
tree.fit(X_train, y_train)
tree_train = accuracy_score(y_train, tree.predict(X_train))
tree_val = accuracy_score(y_val, tree.predict(X_val))
print(f"Tree (max_depth=3)   train={tree_train:.3f}  val={tree_val:.3f}  <- much smaller gap")

In [ ]:
# Draw the shallow tree: the computer WROTE these yes/no rules from the data.
plt.figure(figsize=(18, 8))
plot_tree(
    tree,
    feature_names=list(X_train.columns),
    class_names=["Died", "Survived"],
    filled=True,        # color by predicted class
    rounded=True,
    fontsize=10,
)
plt.title("Decision tree (depth 3): the rules the model learned")
plt.show()

> 🔎 **What to trace**
> - The **first split** the tree chooses is on `Sex` (or the `Title_Mr` flag). It independently rediscovered that sex is the strongest signal.
> - Follow a path of yes/no answers down to a leaf; the leaf's color and `class=` give the prediction for anyone matching that path.
> - The unrestricted tree scored ~0.99 on train but far lower on validation; the depth-3 tree scores similarly on both.
>
> Limiting depth trades a little training accuracy for a lot of **generalization**. Being able to read the model is a bonus, since you can sanity-check its rules against the EDA.

> ✅ **Check your understanding.** The unrestricted tree scored ~0.99 on training but only ~0.78 on validation. What is that gap called, and why does the shallow tree have a smaller one?
> <details><summary>Show answer</summary>
>
> The gap is **overfitting**: the deep tree memorized individual training passengers, so it looks great on data it has seen but generalizes worse. Capping depth at 3 forces broad rules, so training and validation accuracy stay close.
> </details>

**Model 3: Random Forest.** A forest trains many different decision trees on random subsets and lets them vote. Averaging many trees usually generalizes better than any single tree. It also reports `feature_importances_`: how much each feature contributed to its decisions.

> **In plain English: what a random forest is.** A single decision tree can be twitchy, because it may latch onto quirks that are only true for these exact passengers. A random forest calms that down by growing many trees (200 here), each trained on a slightly different random slice of the rows and columns, and then letting them vote. The majority answer wins. Averaging a whole crowd of so-so trees usually beats trusting any single one, the same way polling 200 people gives a steadier answer than asking just one. The trade-off is that a forest is no longer a single readable flowchart.
> Learn more: [StatQuest: Random Forests Part 1 - Building, Using and Evaluating](https://www.youtube.com/watch?v=J4Wdy0Wc_xQ), [scikit-learn: Ensembles - random forests, bagging, boosting](https://scikit-learn.org/stable/modules/ensemble.html)

Just like we capped the single tree at `max_depth=3` so it could not memorize, we cap the depth of the forest's trees (`max_depth=5`) so it generalizes instead of overfitting. Without a depth cap, a forest of unrestricted trees still shows a visible train-validation gap, though averaging across the trees makes it milder than a single unrestricted tree's. We also grow `n_estimators=200` trees so the vote is stable. Watch the train and validation numbers below land close together.

In [ ]:
# Model 3: a Random Forest (many trees voting).
# max_depth=5 caps each tree so the forest generalizes (small train-val gap),
# just like capping the single tree above. n_estimators=200 = 200 trees voting.
forest = RandomForestClassifier(max_depth=5, n_estimators=200, random_state=RANDOM_STATE)
forest.fit(X_train, y_train)
forest_train = accuracy_score(y_train, forest.predict(X_train))
forest_val = accuracy_score(y_val, forest.predict(X_val))
print(f"Random Forest        train={forest_train:.3f}  val={forest_val:.3f}")

In [ ]:
# Which features did the forest lean on most? Sort and plot them.
importances = pd.Series(forest.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=True)   # ascending -> biggest on top in barh
importances.plot(kind="barh")
plt.title("Random Forest feature importances")
plt.xlabel("Relative importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

> 🔎 **What to observe**
> - The sex and title features (`Sex`, `Title_Mr`, …), `Fare`, `Age`, and `Pclass` dominate the importances.
> - The `Embarked_*` dummies and `IsAlone` contribute little, matching the EDA, where the port effect was mostly a class confounder.
> - `Fare` ranks high partly because it stands in for class.
>
> The model's own ranking agrees with what we saw while exploring the data. But "useful for splitting" is not the same as "causes survival," so the same correlation-versus-causation caution applies.

> **What feature importance does and does not tell you.** Once a forest is trained, it can report how much each column helped it split the data, which is handy for spotting which clues carried weight. But even a column of pure random noise picks up a non-zero score (the stretch goal at the end proves it). Read importances as hints, not proof.

Let's put all the models in one table and chart train versus validation side by side, so overfitting jumps out visually.

In [ ]:
# Collect every model's train/val accuracy into a tidy table.
results = pd.DataFrame(
    {
        "Model": ["Baseline", "LogReg", "Tree (full)", "Tree (depth 3)", "Random Forest"],
        "Train": [baseline_acc, logreg_train, full_train, tree_train, forest_train],
        "Validation": [baseline_acc, logreg_val, full_val, tree_val, forest_val],
    }
).set_index("Model")

print(results.round(3))

# Grouped bar: two bars (train, val) per model.
results.plot(kind="bar")
plt.title("Training vs validation accuracy by model")
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.axhline(baseline_acc, color="gray", linestyle="--", label="baseline")
plt.legend()
plt.tight_layout()
plt.show()

> 🔎 **Compare the bars**
> - Every real model clears the dashed baseline line, so they all learned something.
> - **LogReg** (train ~0.83, val ~0.83) and the **depth-3 tree** (train ~0.83, val ~0.83) are both high, and their two bars sit right on top of each other with almost no gap.
> - **Tree (full)** is the outlier: a tall train bar (~0.99) next to a much shorter validation bar (~0.78). That tall-then-short shape is classic overfitting.
> - The **depth-limited Random Forest** (train ~0.85, val ~0.82) is also high, with only a small gap between its two bars, because we capped its depth just like the depth-3 tree.
>
> Prefer the models whose two bars are both high **and** close together (LogReg, the depth-3 tree, and the depth-limited forest). Only the unrestricted tree breaks the pattern, and its big train-minus-validation gap is a warning, not a trophy.

A single split is just **one opinion**: a lucky or unlucky 20% could flatter or punish a model. **Cross-validation** repeats the train/score split several times over different slices and averages the scores. `cross_val_score(..., cv=5)` does 5 rounds; we report the mean and the spread.

> **Why one split is not enough.** A single 80/20 split is one roll of the dice. A lucky split can make a weak model look strong, and an unlucky one can punish a good model. Cross-validation rerolls: it carves the training data into five parts, then trains and scores five times so every part gets a turn as the test slice, and reports the average plus how much the five scores wobble. You end up with an honest range instead of one number you might have gotten lucky on.
> Learn more: [StatQuest: Machine Learning Fundamentals - Cross Validation](https://www.youtube.com/watch?v=fSytzGwwBVw), [scikit-learn: Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html)

In [ ]:
# 5-fold cross-validation of the Random Forest on the training split.
cv_scores = cross_val_score(forest, X_train, y_train, cv=5)
print("5-fold accuracies:", np.round(cv_scores, 3))
print(f"Mean = {cv_scores.mean():.3f}  ±  {cv_scores.std():.3f}")

> 🔎 **What to observe**
> - The five fold scores vary by a few percentage points; that spread is the "how lucky was one split?" wobble.
> - The **mean** sits in the same high-0.7s to low-0.8s range as our single-split numbers.
>
> The model's real performance is a *range* (mean ± spread), not one magic number. Cross-validation gives an honest estimate that depends less on luck.

## Step 9: Evaluate the chosen model

Our selection rule from Step 8 was: prefer a model whose train and validation scores are both high **and** close together. By that rule the **depth-limited Random Forest** (train ~0.85, val ~0.82) qualifies, since it is high with only a small gap, and LogReg (~0.83/~0.83) and the depth-3 tree (~0.83/~0.83) essentially tie it. We pick the Random Forest as our final model because it satisfies the rule *and* gives us `feature_importances_` (used in Step 11); either of the tied models would have been an equally honest choice.

Now we look past a single accuracy number. A **confusion matrix** shows the four possible outcomes: correct deaths, correct survivals, and the two kinds of mistakes.

> **Reading a confusion matrix, precision, and recall.** A single accuracy number hides which kind of mistakes a model makes. The confusion matrix lays out all four cases: correct deaths, correct survivals, false alarms (we said survived, they died), and missed survivors (we said died, they survived). Two questions fall out of it. Precision asks, "of everyone we predicted survived, how many actually did?" Recall asks, "of everyone who actually survived, how many did we catch?" F1 rolls the two into one balanced score. Which one matters most depends on how costly each kind of mistake is.
> Learn more: [StatQuest: Machine Learning Fundamentals - The Confusion Matrix](https://www.youtube.com/watch?v=Kdsp6soqA7o), [Google ML Crash Course: Accuracy, Precision, and Recall](https://developers.google.com/machine-learning/crash-course/classification/accuracy-precision-recall)

In [ ]:
# Predictions of the final model on the held-out validation set.
final_model = forest
val_preds = final_model.predict(X_val)

# Confusion matrix: rows = actual, columns = predicted.
cm = confusion_matrix(y_val, val_preds)

sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Died", "Survived"],
    yticklabels=["Died", "Survived"],
)
plt.title("Confusion matrix (validation set)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

Reading the four quadrants (rows = the truth, columns = the model's guess):

- **Top-left**: actually died, predicted died ✅ (correct).
- **Bottom-right**: actually survived, predicted survived ✅ (correct).
- **Top-right**: actually died, predicted survived ❌ (false alarm).
- **Bottom-left**: actually survived, predicted died ❌ (missed survivor).

> 🔎 **Read the matrix**
> - The two diagonal cells (correct predictions) hold the large majority of the 179 validation passengers.
> - The off-diagonal mistakes are roughly balanced, so the model is not wildly biased toward one class.
>
> Accuracy alone hides *which* mistakes a model makes. The confusion matrix shows both error types, so you can judge whether the balance is acceptable for your use case.

The `classification_report` turns those counts into **precision** (of the passengers we predicted "survived," how many really did?) and **recall** (of the passengers who really survived, how many did we catch?), plus their harmonic mean, the **F1** score.

In [ ]:
# Precision / recall / F1 for each class.
print(classification_report(y_val, val_preds, target_names=["Died", "Survived"]))

Precision and recall trade off against each other, and F1 balances them. Which one matters more depends on the cost of each kind of mistake.

> ⚠️ **Pitfall.** A Titanic model that lands around **0.76-0.82** validation accuracy is doing well. If you ever see **0.90 or higher**, be suspicious: it almost always means leakage (a feature secretly encodes the answer, or you computed fill values before splitting) or overfitting, not genius. Honest beats impressive.

## Step 10: Predict on the test set and write the submission

Finally we predict for the **418** passengers in `test.csv`, which has no `Survived` column. The rule is absolute: apply the exact same preparation we used on training (the same fill values, the same encodings, the same engineered features), then line the columns up to match `X_train`. Because we wrote every cleaning step as a reusable function, we just call them again in the same order.

In [ ]:
# Run the identical pipeline on the raw test data, reusing the TRAIN-derived
# fill values captured back in Step 3 (age_median, fare_median, embarked_mode).
test_processed = basic_clean(test)              # impute + HasCabin
test_processed = encode_sex(test_processed)     # Sex -> 0/1
test_processed = add_family_features(test_processed)  # FamilySize, IsAlone
test_processed = add_title(test_processed)      # Title
test_processed = one_hot(test_processed)        # one-hot Embarked, Title

# Build the test feature matrix and force it to match X_train's columns exactly.
test_X = test_processed.drop(columns=[c for c in drop_cols if c in test_processed.columns])
test_X = test_X.reindex(columns=X_train.columns, fill_value=0)

print("test_X:", test_X.shape, " (should be 418 rows,", X_train.shape[1], "columns)")

Now predict, and assemble the submission file. Kaggle wants exactly two columns, `PassengerId` (from the untouched `test` table) and `Survived` (our predictions), with no index column, so we pass `index=False`.

In [ ]:
# Predict survival for every test passenger.
test_preds = final_model.predict(test_X)

# Build the submission table: PassengerId + our predicted Survived.
submission = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "Survived": test_preds.astype(int),   # ensure clean integers 0/1
    }
)

# Write it next to the notebook. index=False -> no extra unnamed column.
submission.to_csv("submission.csv", index=False)
print("Wrote submission.csv")
submission.head()

A few guard-rail checks so we never submit a malformed file: exactly 418 rows, the two required columns, and integer predictions.

In [ ]:
# Validate the submission shape and dtypes.
assert submission.shape == (418, 2), f"Expected (418, 2), got {submission.shape}"
assert list(submission.columns) == ["PassengerId", "Survived"], "Wrong columns"
assert pd.api.types.is_integer_dtype(submission["Survived"]), "Survived must be integer 0/1"
assert set(submission["Survived"].unique()) <= {0, 1}, "Survived must be only 0 or 1"
print("submission.csv looks good:", submission.shape)
print("Predicted survivors:", int(submission['Survived'].sum()), "of 418")

> 💡 **Optional: submit to the real leaderboard.** Create a free Kaggle account, open the [Titanic - Machine Learning from Disaster](https://www.kaggle.com/c/titanic) competition, and upload this `submission.csv` under **Submit Predictions**. Kaggle scores it against the true (hidden) labels. A score near **0.76-0.79** is a solid, honest result for this workflow.

## Step 11: Stretch goals (optional, off the critical path)

You have completed the full workflow. These extensions are for the curious, and none are required.

1. **Prove that noise cannot help.** Add a column of pure random numbers to `X_train`, retrain the forest, and look at its importance. The demo cell below does exactly this. A useless feature still gets a non-zero importance, which is why we never trust importance blindly.
2. **Smarter age filling.** Instead of one global median, fill missing `Age` with the median age within each `Title` (Masters are boys, Mrs are adults). Remember to still compute those medians from the training split only.
3. **Tune the forest.** Try `n_estimators=300` or `max_depth=6` and see whether cross-validation improves. Change one setting at a time.
4. **Wrap it in a Pipeline.** Combine a `ColumnTransformer` (imputers + one-hot encoders) with the model in a scikit-learn `Pipeline` so preprocessing and modeling are one leakage-proof object.

In [ ]:
# Stretch goal 1, runnable: does a random-noise feature help? (It shouldn't.)
rng = np.random.default_rng(RANDOM_STATE)          # local generator, reproducible
X_train_noise = X_train.copy()
X_train_noise["random_noise"] = rng.normal(size=len(X_train_noise))  # pure noise

# Refit the SAME final (depth-limited) forest, now with the junk column added.
noise_forest = RandomForestClassifier(max_depth=5, n_estimators=200, random_state=RANDOM_STATE)
noise_forest.fit(X_train_noise, y_train)

noise_importance = noise_forest.feature_importances_[list(X_train_noise.columns).index("random_noise")]
print(f"Importance the forest gave to PURE NOISE: {noise_importance:.3f}")
print("Non-zero, yet the feature is meaningless -> importance != usefulness.")

> 🔎 **Worth noticing**
> - The random-noise column receives an importance of about **0.04**. That is not zero, and not even tiny; it is a moderate slice, on par with some of our genuinely useful columns (`Embarked_*`, `IsAlone`).
> - Yet by construction the column is pure noise, carrying **no** information about survival at all.
>
> A meaningless column can still collect a non-trivial importance, because trees will always find *some* way to split on any column. So importance does not prove a feature is real; here a value near 0.04 would look "useful" if we did not know it was junk. Cross-validation and domain sense are the real tests.

## Recap

You ran the whole supervised-learning workflow end to end:

**Load → Explore → Clean → Encode → Engineer → Select → Split → Train → Evaluate → Predict.**

Key habits to keep: split before you compute anything (the Golden Rule), always beat a baseline, watch the train-versus-validation gap for overfitting, and stay suspicious of scores that look too good. Same workflow, new dataset: that is most of applied machine learning.

### Where to learn more

- **Trees and forests:** [StatQuest: Decision and Classification Trees](https://www.youtube.com/watch?v=_L39rN6gz7Y) and [StatQuest: Random Forests Part 1](https://www.youtube.com/watch?v=J4Wdy0Wc_xQ), plus the scikit-learn guides on [Decision Trees](https://scikit-learn.org/stable/modules/tree.html) and [Ensembles](https://scikit-learn.org/stable/modules/ensemble.html).
- **Logistic regression:** [StatQuest: Logistic Regression, Clearly Explained!!!](https://www.youtube.com/watch?v=yIYKR4sgzI8) walks through the S-curve and the weights.
- **Overfitting and cross-validation:** [Google ML Crash Course: Overfitting](https://developers.google.com/machine-learning/crash-course/overfitting/overfitting), [StatQuest: Bias and Variance](https://www.youtube.com/watch?v=EuBBz3bI-aA), and [StatQuest: Cross Validation](https://www.youtube.com/watch?v=fSytzGwwBVw) on why one split is not enough.
- **Metrics:** [StatQuest: The Confusion Matrix](https://www.youtube.com/watch?v=Kdsp6soqA7o) and [Google ML Crash Course: Accuracy, Precision, and Recall](https://developers.google.com/machine-learning/crash-course/classification/accuracy-precision-recall).
- **Try it for real:** the [Kaggle Titanic competition](https://www.kaggle.com/c/titanic), and [pandas: 10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) for a quick tour of the DataFrame tools we used.